# 22｜生成模型_自編碼器GAN與擴散

用線性欠完備自編碼器的等價案例 PCA 觀察壓縮與重建誤差。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/22_生成模型_自編碼器GAN與擴散.md`，目前 47 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

測試重建/去噪、VAE/VQ、GAN 交替梯度、加噪及解析高斯反向取樣。

保留教材原圖；同概念重畫不保證原書分數相同。架構圖以機制或形狀對照，完整模型案例的範圍在各格說明。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 22-03 | 4、5、6、7、8、9、12 | 課堂小實驗 |
| 22-04 | 3、11 | 執行結果圖 |
| 22-05 | 5、6、7、8、9、10、11、12、13、14、15、16、17、18、19、20、47 | 1. 線性 AE、去噪配對與稀疏懲罰 |
| 22-06 | 21、22、23、24、25、26、27、28、29、47 | 2. VAE 抽樣、KL、插值與離散 codebook |
| 22-07 | 30、31、32、33、34、35、47 | 3. GAN 交替梯度與 mode collapse 的診斷 |
| 22-08 | 36、37、38、39、40、41、42、43、44、45、46、47 | 4. 加噪、噪聲目標與高斯玩具反向取樣 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 22-01｜投影片 22：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 22-02｜投影片 22：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

先保留測試集，PCA 只用訓練影像擬合。展示測試影像與重建；後續玩具 VAE/GAN/擴散機制不能當作同一個生成模型的實測成效。

In [ ]:
# 程式 22-03｜投影片 22：4、5、6、7、8、9、12 頁
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
digits = load_digits()
X = digits.data / 16.0
from sklearn.model_selection import train_test_split
train22, test22 = train_test_split(np.arange(len(X)), test_size=.25, stratify=digits.target, random_state=SEED)
pca = PCA(n_components=12, random_state=SEED).fit(X[train22])
Z = pca.transform(X); recon = pca.inverse_transform(Z)
errors = np.mean((X-recon)**2, axis=1)
idx = [int(test22[0]), int(test22[1]), int(test22[np.argmax(errors[test22])])]
pd.DataFrame({'sample': idx, 'label': digits.target[idx], 'reconstruction_MSE': errors[idx]}).round(4)

## 執行結果圖

In [ ]:
# 程式 22-04｜投影片 22：3、11 頁
fig, axes = plt.subplots(2, 3, figsize=(7, 5))
for col, i in enumerate(idx):
    axes[0,col].imshow(X[i].reshape(8,8), cmap='gray'); axes[0,col].set_title(f'original {digits.target[i]}')
    axes[1,col].imshow(recon[i].reshape(8,8), cmap='gray'); axes[1,col].set_title(f'reconstruction\nMSE={errors[i]:.3f}')
    axes[0,col].axis('off'); axes[1,col].axis('off')
savefig('22_generative_demo')
report('generative_models', {'components': 12, 'test_mean_MSE': errors[test22].mean(), 'samples': idx})

## 1. 線性 AE、去噪配對與稀疏懲罰

PCA 的正交編碼／解碼提供線性 AE 的最佳子空間，權重綁定可直接驗算。去噪以乾淨影像作標籤，不把輸入噪聲當目標。

In [ ]:
# 程式 22-05｜投影片 22：5、6、7、8、9、10、11、12、13、14、15、16、17、18、19、20、47 頁
rng22=np.random.default_rng(SEED);center22=pca.mean_;W22=pca.components_.T
encoded22=(X-center22)@W22;decoded22=encoded22@W22.T+center22
assert np.allclose(decoded22,recon)
noisy_train22=np.clip(X[train22]+rng22.normal(0,.2,X[train22].shape),0,1)
noisy_test22=np.clip(X[test22]+rng22.normal(0,.2,X[test22].shape),0,1)
# Ridge 線性去噪器在 train 學 noisy -> clean；小實驗不是卷積 AE
from sklearn.linear_model import Ridge
denoiser22=Ridge(alpha=1).fit(noisy_train22,X[train22]);clean_pred22=denoiser22.predict(noisy_test22)
fig,axes=plt.subplots(3,5,figsize=(8,5))
for j in range(5):
    for row,arr in enumerate([X[test22[j]],noisy_test22[j],clean_pred22[j]]):axes[row,j].imshow(arr.reshape(8,8),cmap='gray',vmin=0,vmax=1);axes[row,j].axis('off')
for row,label in enumerate(['clean target','noisy input','denoised']):axes[row,0].set_title(label,fontsize=9)
savefig('c22_denoising_pairs')
assert (np.array([.025,.08])>.04).tolist()==[False,True]
rho22=.05;rhohat22=np.linspace(.001,.999,300)
kl_sparse22=rho22*np.log(rho22/rhohat22)+(1-rho22)*np.log((1-rho22)/(1-rhohat22))
plt.plot(rhohat22,kl_sparse22);plt.axvline(rho22,c='r',ls='--');plt.xlabel('mean latent activation');plt.ylabel('sparsity KL');savefig('c22_sparse_kl')
# stacked / convolutional AE 的尺寸介面，不訓練完整網路
print('Stacked AE shapes: (B,64) -> (B,32) -> (B,8) -> (B,32) -> (B,64)')
image_batch22=X[test22[:4]].reshape(4,1,8,8);pooled22=image_batch22.reshape(4,1,4,2,4,2).mean((3,5));upsampled22=np.repeat(np.repeat(pooled22,2,2),2,3)
assert upsampled22.shape==image_batch22.shape
print('Noisy / denoised test MSE:',np.mean((noisy_test22-X[test22])**2),np.mean((clean_pred22-X[test22])**2))

## 2. VAE 抽樣、KL、插值與離散 codebook

VAE 用 mu、logvar 與固定 epsilon 重參數化；這裡驗算抽樣公式與損失。插值使用前面的 PCA decoder 展示 latent 操作，不將它冒充訓練好的 VAE。

In [ ]:
# 程式 22-06｜投影片 22：21、22、23、24、25、26、27、28、29、47 頁
mu22=np.array([1.,0.]);logvar22=np.zeros(2);epsilon22=rng22.normal(size=(5000,2));z22=mu22+np.exp(.5*logvar22)*epsilon22
kl22=.5*np.sum(mu22**2+np.exp(logvar22)-logvar22-1);assert np.isclose(kl22,.5)
assert np.allclose(z22.mean(0),mu22,atol=.05)
print('KL / sample moments:',kl22,z22.mean(0),z22.var(0))
# dz/dmu = 1，固定同一 epsilon 才能數值驗算
step22=1e-5;fixed22=epsilon22[0];sample22=lambda mu:mu+np.exp(.5*logvar22)*fixed22
assert np.allclose((sample22(mu22+step22)-sample22(mu22-step22))/(2*step22),1)
a22,b22=encoded22[test22[:2]];interpolated22=np.array([(1-t)*a22+t*b22 for t in np.linspace(0,1,6)])
fig,axes=plt.subplots(1,6,figsize=(10,2))
for ax,arr in zip(axes,interpolated22@W22.T+center22):ax.imshow(arr.reshape(8,8),cmap='gray');ax.axis('off')
plt.suptitle('Interpolation through PCA decoder (not a trained VAE)');savefig('c22_latent_interpolation')
# Gumbel softmax 與硬選取；此 NumPy 範例未實作 straight-through backward
logits22=np.array([.2,.5,1.]);u22=np.clip(rng22.uniform(size=3),1e-8,1-1e-8);gumbel22=-np.log(-np.log(u22))
for temp22 in [1.,.1]:
    p22=np.exp((logits22+gumbel22-(logits22+gumbel22).max())/temp22);p22/=p22.sum();print('Gumbel temperature / soft code:',temp22,p22)
book22=rng22.normal(size=(8,2));encoded_points22=rng22.normal(size=(5,2));ids22=np.sum((encoded_points22[:,None]-book22[None])**2,axis=2).argmin(1)
quantized22=book22[ids22];assert quantized22.shape==encoded_points22.shape
print('VQ nearest IDs / commitment distance:',ids22,np.mean((encoded_points22-quantized22)**2))

## 3. GAN 交替梯度與 mode collapse 的診斷

一維仿射 generator 與 logistic discriminator 展示兩段更新。D 階段固定 G；G 階段仍需要 D 對輸入的導數。右圖以刻意退化的 generator 展示 mode collapse 的判讀，不宣稱是此訓練必然結果。

In [ ]:
# 程式 22-07｜投影片 22：30、31、32、33、34、35、47 頁
sig22=lambda z:1/(1+np.exp(-np.clip(z,-40,40)))
g_a22,g_b22,d_w22,d_b22=1.,0.,.2,0.;trace22=[]
for it22 in range(200):
    noise22=rng22.normal(size=128);real22=rng22.normal(2,.5,128);fake22=g_a22*noise22+g_b22
    pr22=sig22(d_w22*real22+d_b22);pf22=sig22(d_w22*fake22+d_b22)
    old_g22=(g_a22,g_b22);d_w22-=.05*np.mean((pr22-1)*real22+pf22*fake22);d_b22-=.05*np.mean(pr22-1+pf22)
    assert old_g22==(g_a22,g_b22)
    # 非飽和 generator loss -log D(G(z)) 的梯度，D 參數不更新
    pf22=sig22(d_w22*fake22+d_b22);dx22=(pf22-1)*d_w22
    old_d22=(d_w22,d_b22);g_a22-=.05*np.mean(dx22*noise22);g_b22-=.05*np.mean(dx22)
    assert old_d22==(d_w22,d_b22)
    trace22.append([g_a22,g_b22])
fig,axes=plt.subplots(1,2,figsize=(11,3.5));axes[0].plot(trace22);axes[0].set(xlabel='alternating update',ylabel='G parameter',title='Toy GAN updates');axes[0].legend(['scale','shift'])
bimodal22=np.r_[rng22.normal(-2,.3,500),rng22.normal(2,.3,500)];collapsed22=np.full(1000,2.)
axes[1].hist(bimodal22,bins=40,alpha=.6,density=True,label='two real modes');axes[1].axvline(2.,color='tab:orange',lw=3,label='collapsed point mass');axes[1].legend();savefig('c22_gan_collapse')
print('Final toy generator scale / shift:',g_a22,g_b22)

## 4. 加噪、噪聲目標與高斯玩具反向取樣

先用 digits 示範任意 t 加噪，再對已知一維高斯分布使用解析的條件噪聲預測器，執行 DDPM／確定式 DDIM 更新。解析去噪器不是已訓練 U-Net，不能把生成結果稱為影像擴散模型。

In [ ]:
# 程式 22-08｜投影片 22：36、37、38、39、40、41、42、43、44、45、46、47 頁
T22=100;betas22=np.linspace(.0001,.08,T22);alphas22=1-betas22;abar22=np.cumprod(alphas22)
base22=X[test22[0]].reshape(8,8);shared_noise22=rng22.normal(size=(8,8));fig,axes=plt.subplots(1,5,figsize=(10,2.5))
for ax,t22 in zip(axes,[0,20,40,60,99]):
    noisy22=np.sqrt(abar22[t22])*base22+np.sqrt(1-abar22[t22])*shared_noise22
    ax.imshow(noisy22,cmap='gray');ax.set_title(f't={t22}');ax.axis('off')
savefig('c22_forward_noise')
plt.plot(abar22,label='remaining signal power');plt.plot(1-abar22,label='noise power');plt.xlabel('t');plt.legend();savefig('c22_noise_schedule')
mu_data22,sigma_data22=2.,.5
def epsilon_cond22(x,t):
    a=abar22[t];return np.sqrt(1-a)/(a*sigma_data22**2+1-a)*(x-np.sqrt(a)*mu_data22)
samples_ddpm22=rng22.normal(size=4000);samples_ddim22=samples_ddpm22.copy()
for t22 in range(T22-1,-1,-1):
    aprev22=abar22[t22-1] if t22>0 else 1.
    predicted22=epsilon_cond22(samples_ddpm22,t22)
    mean22=(samples_ddpm22-betas22[t22]/np.sqrt(1-abar22[t22])*predicted22)/np.sqrt(alphas22[t22])
    variance22=betas22[t22]*(1-aprev22)/(1-abar22[t22])
    samples_ddpm22=mean22+np.sqrt(variance22)*rng22.normal(size=4000) if t22>0 else mean22
    epshat22=epsilon_cond22(samples_ddim22,t22);x0hat22=(samples_ddim22-np.sqrt(1-abar22[t22])*epshat22)/np.sqrt(abar22[t22])
    samples_ddim22=np.sqrt(aprev22)*x0hat22+np.sqrt(1-aprev22)*epshat22
plt.hist(samples_ddpm22,bins=45,density=True,alpha=.5,label='DDPM');plt.hist(samples_ddim22,bins=45,density=True,alpha=.5,label='DDIM eta=0');plt.legend();plt.xlabel('generated scalar');savefig('c22_reverse_gaussian')
print('Target mean/std:',mu_data22,sigma_data22,'DDPM:',samples_ddpm22.mean(),samples_ddpm22.std(),'DDIM:',samples_ddim22.mean(),samples_ddim22.std())
assert np.isfinite(samples_ddpm22).all()
# 條件生成介面：噪聲 latent、時間與條件共同作輸入
latent_input22=np.c_[encoded22[test22[:4]],np.full(4,.5),np.eye(10)[digits.target[test22[:4]]]]
print('latent + normalized time + condition shape:',latent_input22.shape)

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。